# 📊 ETH OHLCV Data Exploration
## Using SQLite Database

This notebook explores Ethereum OHLCV (Open, High, Low, Close, Volume) data stored in SQLite database.

### 1. Import Libraries and Setup

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import json
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

### 2. Connect to SQLite Database and Load Data

In [ ]:
# Connect to SQLite database
DATA_DIR = Path('..') / 'data'
db_path = DATA_DIR / 'ETH.db'

# Create connection
conn = sqlite3.connect(db_path)

# Check if table exists
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print("📋 Tables in database:")
display(tables)

# Load data from SQLite
query = "SELECT * FROM eth_ohlcv ORDER BY date"
df = pd.read_sql(query, conn, parse_dates=['date'])

# Close connection
conn.close()

print(f"\n✅ Loaded {len(df)} days of OHLCV data from SQLite")
print(f"📅 Period: {df['date'].min().strftime('%Y-%m-%d')} to {df['date'].max().strftime('%Y-%m-%d')}")
df.head(10)

### 3. Basic Statistics from SQLite

In [ ]:
# Reconnect for SQL queries
conn = sqlite3.connect(db_path)

# Get basic statistics using SQL
stats_query = """
SELECT 
    COUNT(*) as total_days,
    MIN(date) as start_date,
    MAX(date) as end_date,
    ROUND(AVG(close), 2) as avg_price,
    ROUND(MIN(close), 2) as min_price,
    ROUND(MAX(close), 2) as max_price,
    ROUND(AVG(volume), 0) as avg_volume,
    ROUND(SUM(volume), 0) as total_volume
FROM eth_ohlcv
"""
stats = pd.read_sql(stats_query, conn)
print("📊 Database Statistics:")
print("="*50)
display(stats)

# Close connection
conn.close()

### 4. Data Statistics with Pandas

In [ ]:
print("📊 Detailed Statistics:")
print("="*50)
display(df[['open', 'high', 'low', 'close', 'volume']].describe())

### 5. Check for Missing Values

In [ ]:
print("🔍 Missing Values:")
missing = df.isnull().sum()
display(missing)

### 6. Correlation Matrix

In [ ]:
# Correlation matrix
correlation = df[['open', 'high', 'low', 'close', 'volume']].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0, fmt='.2f', square=True)
plt.title('📈 Correlation Matrix - ETH Price Metrics', fontsize=14)
plt.tight_layout()
plt.show()

### 7. Price Distribution Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Close price distribution
axes[0, 0].hist(df['close'], bins=50, edgecolor='black', alpha=0.7, color='blue')
axes[0, 0].set_title('Close Price Distribution')
axes[0, 0].set_xlabel('Price (USD)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(df['close'].mean(), color='red', linestyle='--', 
                   label=f'Mean: ${df["close"].mean():.2f}')
axes[0, 0].legend()

# Volume distribution
axes[0, 1].hist(df['volume'], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[0, 1].set_title('Volume Distribution')
axes[0, 1].set_xlabel('Volume')
axes[0, 1].set_ylabel('Frequency')

# Box plot for price
axes[1, 0].boxplot(df['close'])
axes[1, 0].set_title('Close Price Box Plot')
axes[1, 0].set_ylabel('Price (USD)')

# Box plot for volume
axes[1, 1].boxplot(df['volume'])
axes[1, 1].set_title('Volume Box Plot')
axes[1, 1].set_ylabel('Volume')

plt.tight_layout()
plt.show()

### 8. Daily Returns Analysis

In [ ]:
# Calculate returns
df['returns'] = df['close'].pct_change() * 100
df['log_returns'] = np.log(df['close'] / df['close'].shift(1)) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Returns over time
axes[0].plot(df['date'], df['returns'], color='blue', alpha=0.7, linewidth=0.8)
axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[0].set_title('Daily Returns (%) Over Time')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Return (%)')
axes[0].grid(True, alpha=0.3)

# Returns distribution
axes[1].hist(df['returns'].dropna(), bins=50, edgecolor='black', alpha=0.7, color='purple')
axes[1].axvline(df['returns'].mean(), color='red', linestyle='--', 
                label=f'Mean: {df["returns"].mean():.2f}%')
axes[1].axvline(df['returns'].median(), color='green', linestyle='--', 
                label=f'Median: {df["returns"].median():.2f}%')
axes[1].set_title('Returns Distribution')
axes[1].set_xlabel('Return (%)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

print("📊 Returns Statistics:")
print(f"   Mean: {df['returns'].mean():.2f}%")
print(f"   Std Dev: {df['returns'].std():.2f}%")
print(f"   Skewness: {df['returns'].skew():.2f}")
print(f"   Kurtosis: {df['returns'].kurtosis():.2f}")
print(f"   Min: {df['returns'].min():.2f}%")
print(f"   Max: {df['returns'].max():.2f}%")

### 9. SQLite Query Examples

In [ ]:
# Reconnect to database
conn = sqlite3.connect(db_path)

# Query: Top 10 highest price days
top_days_query = """
SELECT date, open, high, low, close, volume
FROM eth_ohlcv
ORDER BY close DESC
LIMIT 10
"""
top_days = pd.read_sql(top_days_query, conn)
print("📈 Top 10 Highest Price Days:")
display(top_days)

# Query: Top 10 highest volume days
top_volume_query = """
SELECT date, open, high, low, close, volume
FROM eth_ohlcv
ORDER BY volume DESC
LIMIT 10
"""
top_volume = pd.read_sql(top_volume_query, conn)
print("\n📊 Top 10 Highest Volume Days:")
display(top_volume)

# Query: Monthly averages
monthly_query = """
SELECT 
    strftime('%Y-%m', date) as month,
    COUNT(*) as days,
    ROUND(AVG(close), 2) as avg_price,
    ROUND(MIN(close), 2) as min_price,
    ROUND(MAX(close), 2) as max_price,
    ROUND(AVG(volume), 0) as avg_volume
FROM eth_ohlcv
GROUP BY month
ORDER BY month
"""
monthly = pd.read_sql(monthly_query, conn)
print("\n📅 Monthly Averages:")
display(monthly.tail(12))

conn.close()

### 10. Summary

✅ Data loaded successfully from SQLite
✅ Basic statistics calculated
✅ Missing values checked
✅ Correlation matrix created
✅ Price distribution analyzed
✅ Returns calculated and visualized
✅ SQL queries executed

Total Records: {len(df)}
Date Range: {df['date'].min()} to {df['date'].max()}